In [ ]:
import pandas as pd
import os
import sys

# 1. Файлын замыг хувьсагчид хадгалах
file_path = "data/turshilt ai.xlsx"

# 2. Файл байгаа эсэхийг шалгах
if not os.path.exists(file_path):
    print(f"❌ Алдаа: {file_path} зам дээр файл олдсонгүй!")
    sys.exit()

# 3. Файлыг унших
df_tech = pd.read_excel(file_path, sheet_name='Тех карт')
df_urtug = pd.read_excel(file_path, sheet_name='Sheet4')

# 4. Огноог хөрвүүлэх (Сар бүрээр нэгтгэхэд заавал хэрэгтэй)
df_tech['effective_date'] = pd.to_datetime(df_tech['effective_date'])
df_urtug['effective_date'] = pd.to_datetime(df_urtug['effective_date'])

# 5. Сар бүрээр нэгтгэх (Year-Month багана үүсгэх)
df_tech['year_month'] = df_tech['effective_date'].dt.to_period('M')
df_urtug['year_month'] = df_urtug['effective_date'].dt.to_period('M')

# 6. Өртгийн дундаж тооцох
urtug_average = df_urtug.groupby(['Buteegdehuunii_id', 'year_month'])['urtug_une'].mean().reset_index()
urtug_average.rename(columns={'urtug_une': 'average_urtug'}, inplace=True)

# 7. Хоёр хүснэгтийг нэгтгэх
df_final = pd.merge(df_tech, urtug_average, on=['Buteegdehuunii_id', 'year_month'], how='left')

# 8. Туслах баганыг устгах
df_final = df_final.drop(columns=['year_month'])

# 9. Үр дүнг хэвлэх
print("✅ Өртөг тооцоололт амжилттай дууслаа.")
print(df_final.head())


import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def generate_cost_report_and_graph():
    file_path = "data/turshilt ai.xlsx"
    if not os.path.exists(file_path):
        return "❌ Эх файл олдсонгүй."

    # 1. Өгөгдлийг уншиж, огноог боловсруулах
    df_tech = pd.read_excel(file_path, sheet_name='Тех карт')
    df_urtug = pd.read_excel(file_path, sheet_name='Sheet4')
    
    df_tech['effective_date'] = pd.to_datetime(df_tech['effective_date'])
    df_urtug['effective_date'] = pd.to_datetime(df_urtug['effective_date'])
    df_tech['year_month'] = df_tech['effective_date'].dt.to_period('M')
    df_urtug['year_month'] = df_urtug['effective_date'].dt.to_period('M')

    # 2. Нэгж өртгийн дундаж гаргах
    urtug_avg = df_urtug.groupby(['Buteegdehuunii_id', 'year_month'])['urtug_une'].mean().reset_index()
    urtug_avg.rename(columns={'urtug_une': 'average_urtug'}, inplace=True)

    # 3. НИЙТ ӨРТӨГ БОДОХ
    df_final = pd.merge(df_tech, urtug_avg, on=['Buteegdehuunii_id', 'year_month'], how='left')
    # Мөр бүрийн өртөг = Орц * Нэгж өртөг
    df_final['row_total'] = df_final['Hemjee'] * df_final['average_urtug']/df_final['Грамм']
    
    # Хоолны нийт өртөг (Сар сараар)
    hool_summary = df_final.groupby(['Hoolnii_ner', 'year_month'])['row_total'].sum().reset_index()

    # 4. ТАЙЛАН ГАРГАХ (Excel руу хадгалах)
    output_path = "data/tailan_urtug_2026.xlsx"
    with pd.ExcelWriter(output_path) as writer:
        df_final.to_excel(writer, sheet_name='Дэлгэрэнгүй', index=False)
        hool_summary.to_excel(writer, sheet_name='Хоолны нийт өртөг', index=False)
    print(f"✅ Тайлан хадгалагдлаа: {output_path}")

    # 5. ГРАФИК (Сар бүрийн өртгийн өөрчлөлт)
    plt.figure(figsize=(12, 6))
    plot_data = hool_summary.copy()
    plot_data['year_month'] = plot_data['year_month'].astype(str) # Графикт зориулж текст болгоно
    
    sns.lineplot(data=plot_data, x='year_month', y='row_total', hue='Hoolnii_ner', marker='o')
    plt.title('Хоолны нийт өртгийн сарын өөрчлөлт (МНТ)', fontsize=14)
    plt.xlabel('Хугацаа (Сар)', fontsize=12)
    plt.ylabel('Нийт өртөг (₮)', fontsize=12)
    plt.legend(title='Хоолны нэр', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, linestyle='--', alpha=0.7)
    plt.tight_layout()
    
    graph_path = "data/cost_trend.png"
    plt.savefig(graph_path)
    plt.show()
    
    return hool_summary

# Ажиллуулах
generate_cost_report_and_graph()





import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def generate_final_report():
    file_path = "data/turshilt ai.xlsx"
    if not os.path.exists(file_path):
        print(f"❌ '{file_path}' файл олдсонгүй.")
        return

    # 1. Өгөгдлийг унших
    df_tech = pd.read_excel(file_path, sheet_name='Тех карт')
    df_urtug = pd.read_excel(file_path, sheet_name='Sheet4')
    df_sales = pd.read_excel(file_path, sheet_name='Борлуулалт')
    df_prod = pd.read_excel(file_path, sheet_name='Үйлдвэрлэл')

    # Огноог сар болгон хөрвүүлэх
    for df in [df_tech, df_urtug, df_sales, df_prod]:
        if 'effective_date' in df.columns:
            df['effective_date'] = pd.to_datetime(df['effective_date'])
            df['year_month'] = df['effective_date'].dt.to_period('M')

    # 2. Түүхий эдийн сарын дундаж үнэ
    urtug_avg = df_urtug.groupby(['Buteegdehuunii_id', 'year_month'])['urtug_une'].mean().reset_index()
    urtug_avg.rename(columns={'urtug_une': 'average_urtug'}, inplace=True)

    # 3. 1 порцны өртөг (Таны оруулсан /Грамм томьёогоор)
    df_unit = pd.merge(df_tech, urtug_avg, on=['Buteegdehuunii_id', 'year_month'], how='left')
    df_unit['row_cost'] = (df_unit['Hemjee'] * df_unit['average_urtug']) / df_unit['Грамм']
    
    hool_unit_cost = df_unit.groupby(['Hoolnii_ner', 'year_month'])['row_cost'].sum().reset_index()
    hool_unit_cost.rename(columns={'row_cost': 'unit_cost'}, inplace=True)

    # 4. Борлуулалт ба Үйлдвэрлэлийг нэгтгэж тооцох
    sales_agg = df_sales.groupby(['Hoolnii_ner', 'year_month']).agg({
        'Sales_Count': 'sum',
        'Sales_Price': 'mean'
    }).reset_index()

    prod_agg = df_prod.groupby(['Hoolnii_ner', 'year_month'])['Production_Count'].sum().reset_index()

    # Сүүлийн нэгдсэн хүснэгт
    final_df = pd.merge(sales_agg, prod_agg, on=['Hoolnii_ner', 'year_month'], how='outer')
    final_df = pd.merge(final_df, hool_unit_cost, on=['Hoolnii_ner', 'year_month'], how='left')

    # 5. САНХҮҮГИЙН ҮЗҮҮЛЭЛТҮҮД
    final_df['Revenue'] = final_df['Sales_Count'] * final_df['Sales_Price']
    final_df['Total_Cost'] = final_df['Production_Count'] * final_df['unit_cost']
    final_df['Profit'] = final_df['Revenue'] - final_df['Total_Cost']
    final_df['Waste_Loss'] = (final_df['Production_Count'] - final_df['Sales_Count']) * final_df['unit_cost']

    # 6. EXCEL-Д ХАДГАЛАХ
    output_path = "tailan_2026_final.xlsx"
    with pd.ExcelWriter(output_path) as writer:
        final_df.to_excel(writer, sheet_name='Нийт тайлан', index=False)
        df_unit.to_excel(writer, sheet_name='Өртгийн дэлгэрэнгүй', index=False)

    print(f"✅ Амжилттай! Тайлан хадгалагдлаа: {output_path}")

    # 7. ГРАФИК ХАРАХ
    plt.figure(figsize=(12, 6))
    plt.style.use('seaborn-v0_8')
    plot_data = final_df.copy()
    plot_data['year_month'] = plot_data['year_month'].astype(str)
    
    sns.barplot(data=plot_data, x='Hoolnii_ner', y='Profit', hue='year_month')
    plt.title('Хоол тус бүрийн бодит ашиг (₮)', fontsize=14)
    plt.ylabel('Ашиг (₮)')
    plt.xlabel('Хоолны нэр')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Дуудах
generate_final_report()




import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

def generate_final_report():
    file_path = "data/turshilt ai.xlsx"
    if not os.path.exists(file_path):
        print(f"❌ '{file_path}' файл олдсонгүй.")
        return

    # 1. Өгөгдлийг унших
    df_tech = pd.read_excel(file_path, sheet_name='Тех карт')
    df_urtug = pd.read_excel(file_path, sheet_name='Sheet4')
    df_sales = pd.read_excel(file_path, sheet_name='Борлуулалт')
    df_prod = pd.read_excel(file_path, sheet_name='Үйлдвэрлэл')

    # Огноог сар болгон хөрвүүлэх
    for df in [df_tech, df_urtug, df_sales, df_prod]:
        if 'effective_date' in df.columns:
            df['effective_date'] = pd.to_datetime(df['effective_date'])
            df['year_month'] = df['effective_date'].dt.to_period('M')

    # 2. Түүхий эдийн сарын дундаж үнэ
    urtug_avg = df_urtug.groupby(['Buteegdehuunii_id', 'year_month'])['urtug_une'].mean().reset_index()
    urtug_avg.rename(columns={'urtug_une': 'average_urtug'}, inplace=True)

    # 3. 1 порцны өртөг (Таны оруулсан /Грамм томьёогоор)
    df_unit = pd.merge(df_tech, urtug_avg, on=['Buteegdehuunii_id', 'year_month'], how='left')
    df_unit['row_cost'] = (df_unit['Hemjee'] * df_unit['average_urtug']) / df_unit['Грамм']
    
    hool_unit_cost = df_unit.groupby(['Hoolnii_ner', 'year_month'])['row_cost'].sum().reset_index()
    hool_unit_cost.rename(columns={'row_cost': 'unit_cost'}, inplace=True)

    # 4. Борлуулалт ба Үйлдвэрлэлийг нэгтгэж тооцох
    sales_agg = df_sales.groupby(['Hoolnii_ner', 'year_month']).agg({
        'Sales_Count': 'sum',
        'Sales_Price': 'mean'
    }).reset_index()

    prod_agg = df_prod.groupby(['Hoolnii_ner', 'year_month'])['Production_Count'].sum().reset_index()

    # Сүүлийн нэгдсэн хүснэгт
    final_df = pd.merge(sales_agg, prod_agg, on=['Hoolnii_ner', 'year_month'], how='outer')
    final_df = pd.merge(final_df, hool_unit_cost, on=['Hoolnii_ner', 'year_month'], how='left')

    # 5. САНХҮҮГИЙН ҮЗҮҮЛЭЛТҮҮД
    final_df['Revenue'] = final_df['Sales_Count'] * final_df['Sales_Price']
    final_df['Total_Cost'] = final_df['Production_Count'] * final_df['unit_cost']
    final_df['Profit'] = final_df['Revenue'] - final_df['Total_Cost']
    final_df['Waste_Loss'] = (final_df['Production_Count'] - final_df['Sales_Count']) * final_df['unit_cost']

    # 6. EXCEL-Д ХАДГАЛАХ
    output_path = "tailan_2026_final.xlsx"
    with pd.ExcelWriter(output_path) as writer:
        final_df.to_excel(writer, sheet_name='Нийт тайлан', index=False)
        df_unit.to_excel(writer, sheet_name='Өртгийн дэлгэрэнгүй', index=False)

    print(f"✅ Амжилттай! Тайлан хадгалагдлаа: {output_path}")

    # 7. ГРАФИК ХАРАХ
    plt.figure(figsize=(12, 6))
    plt.style.use('seaborn-v0_8')
    plot_data = final_df.copy()
    plot_data['year_month'] = plot_data['year_month'].astype(str)
    
    sns.barplot(data=plot_data, x='Hoolnii_ner', y='Profit', hue='year_month')
    plt.title('Хоол тус бүрийн бодит ашиг (₮)', fontsize=14)
    plt.ylabel('Ашиг (₮)')
    plt.xlabel('Хоолны нэр')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

# Дуудах
generate_final_report()





import pandas as pd
import os

def process_waste_and_sales():
    file_path = "data/turshilt ai.xlsx"
    if not os.path.exists(file_path):
        print(f"❌ '{file_path}' файл олдсонгүй.")
        return

    # 1. Өгөгдлийг унших
    xls = pd.ExcelFile(file_path)
    df_tech = pd.read_excel(file_path, sheet_name='Тех карт')
    df_urtug = pd.read_excel(file_path, sheet_name='Sheet4')
    df_sales = pd.read_excel(file_path, sheet_name='Борлуулалт')
    df_prod = pd.read_excel(file_path, sheet_name='Үйлдвэрлэл')

    # Огноог сар болгон хөрвүүлэх (Сар бүрийн өртөг өөр байж болох тул)
    for df in [df_tech, df_urtug, df_sales, df_prod]:
        if 'effective_date' in df.columns:
            df['effective_date'] = pd.to_datetime(df['effective_date'])
            df['year_month'] = df['effective_date'].dt.to_period('M')

    # 2. Түүхий эдийн сарын дундаж үнэ (Sheet4-өөс)
    urtug_avg = df_urtug.groupby(['Buteegdehuunii_id', 'year_month'])['urtug_une'].mean().reset_index()
    urtug_avg.rename(columns={'urtug_une': 'average_urtug'}, inplace=True)

    # 3. Хоолны 1 порцны өртөг (Тех карт-аас row_total тооцох)
    df_unit_calc = pd.merge(df_tech, urtug_avg, on=['Buteegdehuunii_id', 'year_month'], how='left')
    # Томьёо: (Орцны хэмжээ * Түүхий эдийн үнэ) / Тех картны грамм
    df_unit_calc['row_total'] = (df_unit_calc['Hemjee'] * df_unit_calc['average_urtug']) / df_unit_calc['Грамм']
    
    # Сар бүрээр нэг порцны нийт өртөгийг нэгтгэх
    hool_monthly_cost = df_unit_calc.groupby(['Hoolnii_ner', 'year_month'])['row_total'].sum().reset_index()
    hool_monthly_cost.rename(columns={'row_total': 'unit_cost_val'}, inplace=True)

    # 4. Борлуулалт дээрх хоосон утгыг нөхөх
    # Борлуулалтын датаг өртгийн дататай 'Hoolnii_ner' болон 'year_month'-оор холбоно
    df_sales_fixed = pd.merge(df_sales, hool_monthly_cost, on=['Hoolnii_ner', 'year_month'], how='left')

    # Хэрэв Sales_Price багана байхгүй бол (заримдаа нэр зөрөх тохиолдолд) шалгана
    target_col = 'Sales_Price' 
    if target_col in df_sales_fixed.columns:
        # Хоосон (NaN) утгуудыг Тех картнаас тооцоолсон өртгөөр (unit_cost_val) нөхнө
        df_sales_fixed[target_col] = df_sales_fixed[target_col].fillna(df_sales_fixed['unit_cost_val'])
    else:
        print(f"⚠️ Анхаар: 'Борлуулалт' хуудас дээр '{target_col}' багана олдсонгүй.")

    # 5. Эцсийн тайлан бэлтгэх (Үйлдвэрлэлтэй нэгтгэх)
    prod_agg = df_prod.groupby(['Hoolnii_ner', 'year_month'])['Production_Count'].sum().reset_index()
    sales_agg = df_sales_fixed.groupby(['Hoolnii_ner', 'year_month']).agg({
        'Sales_Count': 'sum',
        'Sales_Price': 'mean'
    }).reset_index()

    final_report = pd.merge(sales_agg, prod_agg, on=['Hoolnii_ner', 'year_month'], how='outer')
    final_report = pd.merge(final_report, hool_monthly_cost, on=['Hoolnii_ner', 'year_month'], how='left')

    # Ашиг, орлогын тооцоо
    final_report['Total_Revenue'] = final_report['Sales_Count'] * final_report['Sales_Price']
    final_report['Total_Cost'] = final_report['Production_Count'] * final_report['unit_cost_val']
    final_report['Final_Profit'] = final_report['Total_Revenue'] - final_report['Total_Cost']

    # 6. Файлд хадгалах
    output_file = "tailan_waste_processed.xlsx"
    with pd.ExcelWriter(output_file) as writer:
        final_report.to_excel(writer, sheet_name='Эцсийн тайлан', index=False)
        df_sales_fixed.to_excel(writer, sheet_name='Зассан борлуулалт', index=False)

    print(f"✅ Амжилттай! Хоосон Sales_Price-ийг өртгөөр нөхөж дууслаа: {output_file}")

# Ажиллуулах
process_waste_and_sales()





import pandas as pd
import os

def generate_comprehensive_margin_report(file_path):
    if not os.path.exists(file_path):
        print(f"❌ '{file_path}' файл олдсонгүй.")
        return

    # 1. Өгөгдлийг унших
    xls = pd.ExcelFile(file_path)
    df_tech = pd.read_excel(file_path, sheet_name='Тех карт')
    df_urtug = pd.read_excel(file_path, sheet_name='Sheet4')
    df_sales = pd.read_excel(file_path, sheet_name='Борлуулалт')
    df_prod = pd.read_excel(file_path, sheet_name='Үйлдвэрлэл')

    # Огноог сар болгон хөрвүүлэх (Сар бүрийн өртөг өөр байж болох тул)
    for df in [df_tech, df_urtug, df_sales, df_prod]:
        if 'effective_date' in df.columns:
            df['effective_date'] = pd.to_datetime(df['effective_date'])
            df['year_month'] = df['effective_date'].dt.to_period('M')

    # 2. Өртөг тооцох (Түүхий эдийн дундаж үнэ)
    urtug_avg = df_urtug.groupby(['Buteegdehuunii_id', 'year_month'])['urtug_une'].mean().reset_index()
    urtug_avg.rename(columns={'urtug_une': 'average_urtug'}, inplace=True)
    
    # 3. 1 порцны өртөг (Тех карт-ын логикоор)
    df_unit_calc = pd.merge(df_tech, urtug_avg, on=['Buteegdehuunii_id', 'year_month'], how='left')
    df_unit_calc['row_total'] = (df_unit_calc['Hemjee'] * df_unit_calc['average_urtug']) / df_unit_calc['Грамм']
    
    hool_cost = df_unit_calc.groupby(['Hoolnii_ner', 'year_month'])['row_total'].sum().reset_index()
    hool_cost.rename(columns={'row_total': 'unit_cost'}, inplace=True)

    # 4. Борлуулалтыг засах (Waste тооцох - Хоосон үнийг өртгөөр нөхөх)
    df_sales_fixed = pd.merge(df_sales, hool_cost, on=['Hoolnii_ner', 'year_month'], how='left')
    
    # Sales_Price багана байхгүй бол нэмэх, байвал хоосон утгыг өртгөөр нөхөх
    if 'Sales_Price' in df_sales_fixed.columns:
        df_sales_fixed['Sales_Price'] = df_sales_fixed['Sales_Price'].fillna(df_sales_fixed['unit_cost'])
    else:
        df_sales_fixed['Sales_Price'] = df_sales_fixed['unit_cost']

    # 5. Нэгдсэн тайлан ба Margin тооцоолол
    # Борлуулалт болон Үйлдвэрлэлийг нэгтгэх
    final_report = pd.merge(df_sales_fixed, df_prod, on=['Hoolnii_ner', 'year_month', 'effective_date'], how='outer')
    
    # Өртгийн мэдээллийг дахин баталгаажуулж нэгтгэх
    final_report = pd.merge(final_report, hool_cost, on=['Hoolnii_ner', 'year_month'], how='left', suffixes=('', '_final'))
    if 'unit_cost_final' in final_report.columns:
         final_report['unit_cost'] = final_report['unit_cost'].fillna(final_report['unit_cost_final'])
         final_report.drop(columns=['unit_cost_final'], inplace=True)

    # Margin тооцох
    final_report['Unit_Profit'] = final_report['Sales_Price'] - final_report['unit_cost']
    final_report['Margin_%'] = (final_report['Unit_Profit'] / final_report['Sales_Price']) * 100
    final_report['Food_Cost_%'] = (final_report['unit_cost'] / final_report['Sales_Price']) * 100
    
    # Санхүүгийн нийт дүн
    final_report['Total_Revenue'] = final_report['Sales_Count'] * final_report['Sales_Price']
    final_report['Total_Production_Cost'] = final_report['Production_Count'] * final_report['unit_cost']
    final_report['Net_Profit'] = final_report['Total_Revenue'] - final_report['Total_Production_Cost']

    # 6. Файлд хадгалах
    output_file = "final_margin_analysis_report.xlsx"
    final_report.to_excel(output_file, index=False)
    
    print(f"✅ Амжилттай! Бүх тооцоолол ба Margin орсон тайлан бэлэн боллоо: {output_file}")

# Кодыг ажиллуулах
file_path = "data/turshilt ai.xlsx"
generate_comprehensive_margin_report(file_path)





import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def perform_extended_analysis(file_path):
    xls = pd.ExcelFile(file_path)
    df_tech = pd.read_excel(xls, 'Тех карт')
    df_urtug = pd.read_excel(xls, 'Sheet4')
    df_sales = pd.read_excel(xls, 'Борлуулалт')
    df_prod = pd.read_excel(xls, 'Үйлдвэрлэл')

    # Огноо засах
    for df in [df_tech, df_urtug, df_sales, df_prod]:
        if 'effective_date' in df.columns:
            df['effective_date'] = pd.to_datetime(df['effective_date'])
            df['year_month'] = df['effective_date'].dt.to_period('M').astype(str)

    # 1. Өртөг тооцох
    urtug_avg = df_urtug.groupby(['Buteegdehuunii_id', 'year_month'])['urtug_une'].mean().reset_index()
    urtug_avg.rename(columns={'urtug_une': 'average_urtug'}, inplace=True)
    df_unit_calc = pd.merge(df_tech, urtug_avg, on=['Buteegdehuunii_id', 'year_month'], how='left')
    df_unit_calc['row_total'] = (df_unit_calc['Hemjee'] * df_unit_calc['average_urtug']) / df_unit_calc['Грамм']
    hool_cost = df_unit_calc.groupby(['Hoolnii_ner', 'year_month'])['row_total'].sum().reset_index()
    hool_cost.rename(columns={'row_total': 'unit_cost'}, inplace=True)

    # 2. Нэгдсэн дата үүсгэх (Unit_Profit-ийг энд үүсгэнэ)
    final_report = pd.merge(df_sales, hool_cost, on=['Hoolnii_ner', 'year_month'], how='left')
    final_report['Sales_Price'] = final_report['Sales_Price'].fillna(final_report['unit_cost'])
    final_report['Unit_Profit'] = final_report['Sales_Price'] - final_report['unit_cost']

    # 3. График зурах
    plt.figure(figsize=(12, 6))
    plot_data = final_report.groupby('Hoolnii_ner')['Unit_Profit'].mean().sort_values().reset_index()
    sns.barplot(data=plot_data, x='Hoolnii_ner', y='Unit_Profit', 
                palette=['red' if x < 0 else 'green' for x in plot_data['Unit_Profit']])
    plt.xticks(rotation=90)
    plt.title('Average Profit per Unit')
    plt.savefig('profit_chart.png')
    
    print("✅ Амжилттай: 'profit_chart.png' болон 'inventory_alert.xlsx' үүслээ.")

perform_extended_analysis("data/turshilt ai.xlsx")
